Requires **flowmeter** — install once from the repo root:

```bash
pip install -e tools/flowmeter/
```

# Imports

In [1]:
import os
import sys
import glob
import tempfile
import pandas as pd
from pcapflower import convert_pcap_to_csv

# Inputs

In [2]:
DATA_ROOT = "../data"

# Preprocessing

In [3]:
def find_pcap_folders(root_dir):
    """Return every folder under root_dir that contains at least one PCAP file."""
    found = []
    for dirpath, _, filenames in os.walk(root_dir):
        if any(f.endswith((".pcap", ".pcapng")) for f in filenames):
            found.append(dirpath)
    return sorted(found)


def generate_merged_csv_from_pcaps(pcap_dir, output_filename):
    pcap_files = sorted(
        glob.glob(os.path.join(pcap_dir, "*.pcap")) +
        glob.glob(os.path.join(pcap_dir, "*.pcapng"))
    )

    if not pcap_files:
        raise FileNotFoundError(f"No PCAP files found in: {pcap_dir}")

    label = os.path.basename(pcap_dir)
    print(f"[{label}] {len(pcap_files)} PCAP file(s)")

    output_path = os.path.join(pcap_dir, output_filename)
    total_flows = 0
    first_write = True

    with tempfile.TemporaryDirectory(dir=pcap_dir) as tmp_dir:
        for i, pcap_path in enumerate(pcap_files, 1):
            pcap_name = os.path.basename(pcap_path)
            print(f"  [{i}/{len(pcap_files)}] {pcap_name}")

            tmp_csv = os.path.join(tmp_dir, f"{pcap_name}.csv")
            n_flows = convert_pcap_to_csv(pcap_path, tmp_csv)
            print(f"    → {n_flows} flows")

            if n_flows > 0:
                for chunk in pd.read_csv(tmp_csv, chunksize=50_000):
                    chunk["label"] = label
                    chunk.to_csv(output_path, mode="w" if first_write else "a",
                                 index=False, header=first_write)
                    total_flows += len(chunk)
                    first_write = False

    if total_flows == 0:
        raise RuntimeError(f"No flows extracted from: {pcap_dir}")

    print(f"  ✓ {output_path}  ({total_flows} flows)")
    return output_path

In [ ]:
pcap_folders = find_pcap_folders(DATA_ROOT)
print(f"Found {len(pcap_folders)} folder(s) with PCAPs\n")

for folder_path in pcap_folders:
    folder_name = os.path.basename(folder_path)
    generate_merged_csv_from_pcaps(folder_path, f"merged_{folder_name}.csv")

Found 7 folder(s) with PCAPs

[benign] 1 PCAP file(s)
  [1/1] benign_whole-network3.pcap
    → 22539 flows
  ✓ ../data/raw/CIC_IIoT_dataset_2025/benign/merged_benign.csv  (22539 flows)
[bruteforce] 8 PCAP file(s)
  [1/8] attack_bruteforce_dictionary-ssh_ap.pcap
    → 455 flows
  [2/8] attack_bruteforce_dictionary-ssh_edge1.pcap
    → 492 flows
  [3/8] attack_bruteforce_dictionary-ssh_mqtt-broker.pcap
    → 222 flows
  [4/8] attack_bruteforce_dictionary-ssh_router.pcap
    → 843 flows
  [5/8] attack_bruteforce_dictionary-ssh_switch.pcap
    → 822 flows
  [6/8] attack_bruteforce_dictionary-telnet_ap.pcap
    → 2133 flows
  [7/8] attack_bruteforce_dictionary-telnet_edge1.pcap
    → 969 flows
  [8/8] attack_bruteforce_dictionary-telnet_mqtt-broker.pcap
    → 731 flows
  ✓ ../data/raw/CIC_IIoT_dataset_2025/bruteforce/merged_bruteforce.csv  (6667 flows)
[ddos] 294 PCAP file(s)
  [1/294] attack_ddos_ack-frag-flood-port-1883_accelerometer-sensor.pcap
    → 224458 flows
  [2/294] attack_ddos_ac